# 01 — Environment Setup & BDD100K Data Preparation

This notebook:
1. Installs all dependencies
2. Verifies the GPU and CUDA environment
3. Converts BDD100K per-image JSON labels to YOLO and COCO formats

**Expected input layout on disk:**
```
data/
  images/train/   images/val/   images/test/
  labels/train/   labels/val/   labels/test/   ← one .json per image
```

**Output splits produced:**
- `clear_day/train` and `clear_day/val` — used for training all models
- `rainy`, `snowy`, `night`, `overcast` — used for robustness evaluation

In [ ]:
!pip install -q ultralytics rfdetr fvcore nvidia-ml-py python-dotenv

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
from dotenv import load_dotenv
load_dotenv('../.env')

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
from pathlib import Path

DATA_ROOT     = Path(os.environ.get('BDD100K_ROOT', 'data'))
YOLO_DATA_DIR = Path(os.environ.get('YOLO_DATA_DIR', 'data/bdd100k_yolo'))
COCO_DATA_DIR = Path(os.environ.get('COCO_DATA_DIR', 'data/bdd100k_coco'))

print('BDD100K root exists:', DATA_ROOT.exists())
print('  images/train:', (DATA_ROOT / 'images' / 'train').exists())
print('  labels/train:', (DATA_ROOT / 'labels' / 'train').exists())

In [ ]:
# TODO: If running on Colab, transfer BDD100K from Google Drive or upload:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   !cp /content/drive/MyDrive/bdd100k.zip /content/
#   !unzip -q /content/bdd100k.zip -d /content/

In [ ]:
# Quick sanity check — count JSON files per raw split
for split in ['train', 'val', 'test']:
    lbl_dir = DATA_ROOT / 'labels' / split
    count = len(list(lbl_dir.glob('*.json'))) if lbl_dir.exists() else 0
    print(f'labels/{split}: {count} JSON files')

In [ ]:
# Preview one label file to confirm format
import json
sample = next((DATA_ROOT / 'labels' / 'train').glob('*.json'))
with open(sample) as f:
    d = json.load(f)
print('Attributes:', d.get('attributes'))
print('Frames:', len(d.get('frames', [])))
if d.get('frames'):
    print('Objects in frame 0:', len(d['frames'][0].get('objects', [])))

In [ ]:
from src.data_utils import create_splits, CONDITION_FILTERS

# Convert all splits — may take 10-30 min depending on storage speed.
# Set coco_output=None to skip COCO conversion if not training RF-DETR.
create_splits(
    data_root=DATA_ROOT,
    yolo_output=YOLO_DATA_DIR,
    coco_output=COCO_DATA_DIR,
)
print('Done.')

In [ ]:
# Verify output structure
splits = ['clear_day/train', 'clear_day/val', 'rainy', 'snowy', 'night', 'overcast']
for split in splits:
    img_dir = YOLO_DATA_DIR / split / 'images'
    lbl_dir = YOLO_DATA_DIR / split / 'labels'
    n_img = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'YOLO {split:20s}: {n_img:5d} images  {n_lbl:5d} labels')